# Phase 2X - Cross-validation and a corrected leakage measurement

**Run on:** Kaggle or Colab, free T4. Budget ~60-80 min for 50 training runs.

---

### Why this notebook exists

Two weaknesses in the results so far, both addressed here.

**1. Every previous number came from a single split.** One partition, one seed, no
variance. That is adequate for a project report and thin for a paper: a reviewer will
reasonably ask whether the result is an artefact of one lucky partition. Five-fold
cross-validation replaces point estimates with mean and standard deviation.

**2. The Phase 0 leakage estimate measured the wrong thing.** It took one model, whose
training data was fixed, and evaluated it on an image-level test set and on a grouped
test set. The difference between those numbers conflates two effects: genuine leakage,
and the two test sets simply being of different difficulty.

The correct comparison trains **under** each protocol and evaluates within it:

| | training partition | test partition |
|---|---|---|
| Image-level protocol | images assigned independently | images assigned independently |
| Grouped protocol | whole pseudo-patient groups | whole pseudo-patient groups |

Identical architecture, identical hyper-parameters, identical fold count. The only
difference is whether the partition respects pseudo-patient boundaries. Any gap is then
attributable to the protocol. This mirrors the design used by Yagis et al. for
slice-level splitting in brain MRI.

**Five architectures are evaluated**, chosen so that architecture family and
pretraining vary independently rather than together:

| family | pretrained | model |
|---|---|---|
| transformer | no | ViT-30, the original project design |
| transformer | no, smaller | ViT-small, capacity-matched to 767 images |
| **transformer** | **yes** | **ViT-B/16, ImageNet pretrained** |
| CNN | yes | EfficientNet-B0 (compound scaling, squeeze-excitation) |
| CNN | yes | ResNet50 (plain residual bottlenecks) |

The pretrained transformer is the arm that breaks the confound. Against the CNNs it
holds pretraining constant and varies family; against the from-scratch transformers it
holds family constant and varies pretraining. Without it, a null result on the
from-scratch ViT cannot be attributed to either cause.

The selection is deliberate. Comparing one from-scratch transformer against one
pretrained CNN confounds two variables, so a null result on the transformer cannot be
attributed. Two CNNs of different design test whether any effect is an artefact of one
architecture's inductive biases; two transformers of different capacity separate
"transformers need more data than this" from "this particular transformer was too large
for this dataset".

### Fixed before running
- **No early stopping.** Epoch counts are fixed per architecture, matched to where the
  Phase 2C runs converged, and applied identically across every protocol and fold. Early
  stopping would require a validation split whose construction differs between
  protocols, which would contaminate the comparison it exists to measure.
- **No model selection.** Every configuration is specified in advance and all results
  are reported, so there is no selection effect to correct for.

## 1. Environment and data

In [ ]:
import subprocess, sys
for pkg in ["opencv-python-headless", "tabulate", "kagglehub", "scipy",
            "keras-hub"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg],
                   check=False)

import tensorflow as tf, keras
gpus = tf.config.list_physical_devices("GPU")
print("tensorflow:", tf.__version__, "| keras:", keras.__version__)
print("GPUs      :", gpus)
if not gpus:
    print("\nNO GPU DETECTED - 50 training runs on CPU is not practical.")
    print("  Colab : Runtime > Change runtime type > T4 GPU")
    print("  Kaggle: Settings > Accelerator > GPU (needs a verified phone number)")

In [ ]:
import os, json, time, shutil, random, gc
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             balanced_accuracy_score, confusion_matrix)
from scipy import stats

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

IN_KAGGLE = os.path.exists("/kaggle/working")
WORK    = "/kaggle/working" if IN_KAGGLE else "/content"
SCRATCH = "/kaggle/temp"    if IN_KAGGLE else "/content"
try:
    os.makedirs(SCRATCH, exist_ok=True)
except OSError:
    SCRATCH = "/tmp"; os.makedirs(SCRATCH, exist_ok=True)
print("environment:", "Kaggle" if IN_KAGGLE else "Colab", "| outputs ->", WORK)

def get_secret(name):
    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return (UserSecretsClient().get_secret(name) or "").strip() or None
        from google.colab import userdata
        return (userdata.get(name) or "").strip() or None
    except Exception as e:
        if "NotebookAccess" in type(e).__name__:
            print(f"  {name}: exists but this notebook lacks access")
        return None

def deliver(zip_base, src_dir):
    path = shutil.make_archive(zip_base, "zip", src_dir)
    print("archive:", path, f"({os.path.getsize(path)/1e6:.1f} MB)")
    if not IN_KAGGLE:
        try:
            from google.colab import files; files.download(path)
        except Exception as e:
            print("download it from the file browser:", e)
    else:
        print("Kaggle: it is under /kaggle/working - use the Output panel")

CLASS_NAMES = ["Benign", "Malignant", "Normal"]
FOLDERS = {"Benign": "Bengin cases", "Malignant": "Malignant cases",
           "Normal": "Normal cases"}
N_FOLDS = 5

RESULTS_DIR = f"{WORK}/fyp_phase2x_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
RESULTS = {"seed": SEED, "n_folds": N_FOLDS}

def save_json():
    with open(f"{RESULTS_DIR}/results.json", "w") as f:
        json.dump(RESULTS, f, indent=2, default=float)
print("results ->", RESULTS_DIR)

In [ ]:
# Prefer a dataset attached to the notebook (/kaggle/input). That needs no
# credentials at all, which matters when the kernel is launched via the API and
# has no Secrets attached. Fall back to kagglehub when running elsewhere.
DATA_ROOT = None
if os.path.isdir("/kaggle/input"):
    hits = [d for d, _, _ in os.walk("/kaggle/input")
            if os.path.basename(d) == "Malignant cases"]
    if hits:
        DATA_ROOT = os.path.dirname(hits[0])
        print("using the attached dataset:", DATA_ROOT)

if DATA_ROOT is None:
    if get_secret("KAGGLE_API_TOKEN"):
        os.environ["KAGGLE_API_TOKEN"] = get_secret("KAGGLE_API_TOKEN")
    elif get_secret("KAGGLE_USERNAME") and get_secret("KAGGLE_KEY"):
        os.environ["KAGGLE_USERNAME"] = get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = get_secret("KAGGLE_KEY")
    else:
        raise RuntimeError(
            "No attached dataset and no Kaggle credential. Either attach "
            "hamdallak/the-iqothnccd-lung-cancer-dataset to the notebook, or add "
            "KAGGLE_API_TOKEN to Secrets.")
    import kagglehub
    DL = kagglehub.dataset_download("hamdallak/the-iqothnccd-lung-cancer-dataset")
    cands = [d for d, _, _ in os.walk(DL) if os.path.basename(d) == "Malignant cases"]
    DATA_ROOT = os.path.dirname(cands[0])
    print("downloaded via kagglehub:", DATA_ROOT)

SPLIT_URL = ("https://raw.githubusercontent.com/haseebkhan9081/"
             "iqothnccd-leakage-audit/main/split_seed42.csv")
subprocess.run(["wget", "-q", "-O", f"{SCRATCH}/split_seed42.csv", SPLIT_URL], check=True)
df = pd.read_csv(f"{SCRATCH}/split_seed42.csv")
df["path"] = [os.path.join(DATA_ROOT, FOLDERS[l], f)
              for l, f in zip(df["label"], df["file"])]
assert all(os.path.exists(p) for p in df["path"])

y_all = df["y"].to_numpy()
groups = df["group"].to_numpy()
print("images:", len(df), "| pseudo-patient groups:", df["group"].nunique())
print(df["label"].value_counts().to_string())
RESULTS["data"] = {"n_images": int(len(df)), "n_groups": int(df["group"].nunique())}
save_json()

## 2. The two partitioning protocols

`StratifiedGroupKFold` guarantees that no pseudo-patient group appears in two folds
while still balancing the class distribution across folds. `StratifiedKFold` over
images ignores group membership entirely, which is the convention this paper is
measuring.

Both arms are stratified, deliberately. An earlier version used plain `GroupKFold`
here, which does not stratify; the grouped folds then carried an uncontrolled class
distribution while the image-level folds were balanced by construction, so the
measured gap mixed leakage with a class-distribution shift. Since balanced accuracy
is the metric most sensitive to that shift, and it is the headline metric, the two
arms must agree on stratification for the partitioning to be the only difference.
The cell below records each fold's test-set class prevalence so the claim is
checkable rather than asserted.

In [ ]:
def folds_grouped():
    # StratifiedGroupKFold, NOT GroupKFold. Plain GroupKFold does not stratify,
    # so the grouped arm would carry an uncontrolled class distribution while
    # the image-level arm is stratified by construction. The measured gap would
    # then be leakage PLUS a class-distribution shift, and balanced accuracy is
    # the metric most sensitive to exactly that. Holding stratification fixed
    # across both arms is what makes the partitioning the only difference.
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True,
                                random_state=SEED)
    return list(sgkf.split(df, y_all, groups=groups))

def folds_imagelevel():
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    return list(skf.split(df, y_all))

PROTOCOLS = {"grouped": folds_grouped(), "image-level": folds_imagelevel()}

# Record the per-fold class prevalence of each test partition. If the two
# protocols drift apart here, the comparison is confounded and the numbers
# downstream do not mean what the paper says they mean, so this is measured
# rather than assumed.
RESULTS["test_class_prevalence"] = {
    name: [[round(float((y_all[te] == c).mean()), 4) for c in range(3)]
           for _, te in folds]
    for name, folds in PROTOCOLS.items()}
for name, rows in RESULTS["test_class_prevalence"].items():
    print(f"{name:12s} test-fold class prevalence (benign/malig/normal):")
    for k, r in enumerate(rows):
        print(f"    fold {k}: {r}")
_pop = [float((y_all == c).mean()) for c in range(3)]
print(f"population prevalence: {[round(p, 4) for p in _pop]}")
_max_dev = max(abs(r[c] - _pop[c])
               for rows in RESULTS["test_class_prevalence"].values()
               for r in rows for c in range(3))
print(f"largest deviation from population prevalence: {_max_dev:.4f}")

for name, folds in PROTOCOLS.items():
    spans = [len(set(groups[tr]) & set(groups[te])) for tr, te in folds]
    print(f"{name:12s} groups appearing in BOTH train and test, per fold: {spans}")
print()
print("The grouped protocol must show all zeros. The image-level protocol shows how")
print("many pseudo-patients are split across the boundary, which is the leak itself.")
RESULTS["protocol_check"] = {
    n: [int(len(set(groups[tr]) & set(groups[te]))) for tr, te in f]
    for n, f in PROTOCOLS.items()}
assert all(v == 0 for v in RESULTS["protocol_check"]["grouped"]), \
    "a group spans train and test under the grouped protocol"
save_json()

## 3. Architectures

In [ ]:
def build_vit(size=30, n_classes=3, patch=6, proj=64, heads=8, depth=8):
    resize_to = 72

    class Patches(layers.Layer):
        def __init__(self, ps, **kw):
            super().__init__(**kw); self.ps = ps
        def call(self, x):
            b = tf.shape(x)[0]
            p = tf.image.extract_patches(
                images=x, sizes=[1, self.ps, self.ps, 1],
                strides=[1, self.ps, self.ps, 1], rates=[1, 1, 1, 1], padding="VALID")
            return tf.reshape(p, [b, -1, self.ps * self.ps * x.shape[-1]])

    class PatchEncoder(layers.Layer):
        def __init__(self, n, d, **kw):
            super().__init__(**kw)
            self.n = n; self.proj = layers.Dense(d)
            self.pos = layers.Embedding(input_dim=n, output_dim=d)
        def call(self, x):
            return self.proj(x) + self.pos(tf.range(self.n))

    def mlp(x, units, rate):
        for u in units:
            x = layers.Dropout(rate)(layers.Dense(u, activation=tf.nn.gelu)(x))
        return x

    inp = layers.Input((size, size, 3))
    x = keras.Sequential([
        layers.Rescaling(1.0 / 255.0), layers.Resizing(resize_to, resize_to),
        layers.RandomFlip("horizontal"), layers.RandomRotation(0.05),
        layers.RandomZoom(0.15, 0.15)], name="aug")(inp)
    x = Patches(patch)(x)
    enc = PatchEncoder((resize_to // patch) ** 2, proj)(x)
    for _ in range(depth):
        a = layers.LayerNormalization(epsilon=1e-6)(enc)
        a = layers.MultiHeadAttention(num_heads=heads, key_dim=proj, dropout=0.1)(a, a)
        b = layers.Add()([a, enc])
        c = layers.LayerNormalization(epsilon=1e-6)(b)
        c = mlp(c, [proj * 2, proj], 0.1)
        enc = layers.Add()([c, b])
    r = layers.Flatten()(layers.LayerNormalization(epsilon=1e-6)(enc))
    r = mlp(layers.Dropout(0.5)(r), [2048, 1024], 0.5)
    return keras.Model(inp, layers.Dense(n_classes, activation="softmax")(r))

def _transfer(base_fn, preprocess, size, n_classes=3, unfreeze=20):
    inp = layers.Input((size, size, 3))
    x = keras.Sequential([layers.RandomFlip("horizontal"),
                          layers.RandomRotation(0.05),
                          layers.RandomZoom(0.15, 0.15)], name="aug")(inp)
    base = base_fn(include_top=False, weights="imagenet",
                   input_shape=(size, size, 3))
    base.trainable = True
    for layer in base.layers[:-unfreeze]:
        layer.trainable = False
    x = base(preprocess(x), training=False)
    x = layers.Dropout(0.3)(layers.GlobalAveragePooling2D()(x))
    return keras.Model(inp, layers.Dense(n_classes, activation="softmax")(x))

def build_effnet(size=224, n_classes=3):
    return _transfer(keras.applications.EfficientNetB0,
                     keras.applications.efficientnet.preprocess_input,
                     size, n_classes)

def build_resnet(size=224, n_classes=3):
    # A deliberately different CNN design: plain residual bottlenecks, against
    # EfficientNet's compound scaling, depthwise separable convolutions and
    # squeeze-excitation. If the effect appears in both, it is not an artefact
    # of one architecture's inductive biases.
    return _transfer(keras.applications.ResNet50,
                     keras.applications.resnet50.preprocess_input,
                     size, n_classes)

def build_small_vit(size=72, n_classes=3):
    # Capacity control: 4 layers / 4 heads instead of 8 / 8, sized to what 767
    # images can plausibly support. Separates "transformers need more data than
    # this" from "that particular transformer was too large for this dataset".
    return build_vit(size, n_classes, patch=6, proj=64, heads=4, depth=4)


def build_pretrained_vit(size=224, n_classes=3, unfreeze=20):
    # A genuinely ImageNet-pretrained transformer. This is the arm that breaks
    # the confound: the from-scratch ViT and the pretrained CNN differ in BOTH
    # architecture family and pretraining, so neither alone explains the ViT's
    # collapse. With a pretrained transformer, family is held against the CNNs
    # while pretraining is held against the from-scratch ViTs.
    #
    # Keras `applications` ships no ViT, so this comes from KerasHub. If the
    # preset cannot be fetched the architecture is SKIPPED rather than silently
    # replaced with something else - a substituted model would invalidate the
    # comparison this arm exists to make.
    import keras_hub
    backbone = keras_hub.models.Backbone.from_preset(
        "vit_base_patch16_224_imagenet")

    # Match the CNN arms' training regime. `_transfer` unfreezes the last 20
    # weight-bearing layers of a flat `base.layers`; a KerasHub ViT nests its
    # blocks, so flatten to leaves and unfreeze the same number. Leaving the
    # backbone fully frozen would make this a linear probe while the CNNs are
    # partially fine-tuned, confounding architecture family with how much of
    # the network was allowed to adapt.
    def leaves(layer):
        subs = getattr(layer, "layers", None) or []
        return [layer] if not subs else [x for s in subs for x in leaves(s)]

    weighted = [l for l in leaves(backbone) if l.weights]
    # `weighted[:-unfreeze]` is EMPTY when len(weighted) <= unfreeze, which
    # freezes nothing and silently produces a full fine-tune. That is exactly
    # what happened on the first attempt, because a KerasHub ViT exposes far
    # fewer weight-bearing leaves than a Keras `applications` CNN. Cap the
    # count at a third of the layers so the slice always leaves something
    # frozen, whatever the backbone's internal structure turns out to be.
    k = max(1, min(unfreeze, len(weighted) // 3))
    backbone.trainable = True
    for l in weighted[:-k]:
        l.trainable = False
    # Record the trainable fraction rather than asserting on it. `n_train > 0`
    # passes just as happily when NOTHING was frozen, which is what happened on
    # run v4: the leaf walk found <= `unfreeze` weight-bearing layers, so the
    # slice `weighted[:-unfreeze]` was empty and all 85.8M parameters stayed
    # trainable. Check both ends.
    n_train = sum(int(np.prod(w.shape)) for w in backbone.trainable_weights)
    n_all = sum(int(np.prod(w.shape)) for w in backbone.weights)
    frac = n_train / n_all
    print(f"ViT-B/16: {len(weighted)} weight-bearing layers, last {k} "
          f"trainable -> {n_train:,}/{n_all:,} params ({frac:.1%})")
    assert n_train > 0, "ViT backbone ended up fully frozen"

    # Do NOT assert on the trainable fraction here. This architecture is marked
    # optional, and the loop below catches any exception from an optional build
    # and drops that architecture from the factorial. An assertion would
    # therefore turn a regime mismatch into a SILENTLY MISSING RESULT, which is
    # a worse failure than the one it guards against: re-running would quietly
    # produce 40 runs instead of 50 and no ViT row at all. Record the fraction,
    # warn loudly, and let the analysis report the regime it actually got.
    if frac > 0.95:
        print("  WARNING: the partial unfreeze did not take. Only "
              f"{len(weighted)} weight-bearing leaf layers were found against "
              f"unfreeze={unfreeze}, so the slice froze nothing and this arm "
              "is a FULL fine-tune. Its absolute scores are not comparable "
              "with the partially fine-tuned CNNs. Reported, not suppressed.")

    # ImageNet ViT presets expect [-1, 1], not [0, 1]. Feeding the wrong range
    # to a pretrained backbone silently degrades its features, so take the
    # constants from the preset's own converter and fall back to the ViT
    # default only if that lookup fails.
    nm, ns, src = [0.5] * 3, [0.5] * 3, "ViT default (0.5/0.5)"
    try:
        conv = keras_hub.layers.ImageConverter.from_preset(
            "vit_base_patch16_224_imagenet")
        if getattr(conv, "norm_mean", None) and getattr(conv, "norm_std", None):
            nm, ns, src = list(conv.norm_mean), list(conv.norm_std), "preset converter"
    except Exception as e:
        print(f"  converter lookup failed ({type(e).__name__}), using default")
    print(f"  normalisation mean={nm} std={ns} [{src}]")

    inp = layers.Input((size, size, 3))
    x = keras.Sequential([layers.RandomFlip("horizontal"),
                          layers.RandomRotation(0.05),
                          layers.RandomZoom(0.15, 0.15)], name="aug")(inp)
    x = layers.Rescaling(1.0 / 255.0)(x)
    x = layers.Normalization(mean=nm, variance=[float(s) ** 2 for s in ns])(x)
    feats = backbone(x)
    if len(feats.shape) == 3:           # (batch, tokens, dim) -> pool tokens
        feats = layers.GlobalAveragePooling1D()(feats)
    feats = layers.Dropout(0.3)(feats)
    return keras.Model(inp, layers.Dense(n_classes, activation="softmax")(feats))

# Epochs fixed to where the Phase 2C runs converged. No early stopping: it would
# need a validation split whose construction differs between protocols, which would
# contaminate the very comparison this notebook exists to make.
# Five architectures, chosen so that architecture family and pretraining vary
# separately rather than together:
#
#   family        pretrained    model
#   transformer   no            ViT-30 (the original FYP design)
#   transformer   no, smaller   ViT-small (capacity-matched to 767 images)
#   transformer   yes           ViT-B/16 (ImageNet-21k -> 1k)
#   CNN           yes           EfficientNetB0 (compound scaling, SE blocks)
#   CNN           yes           ResNet50 (plain residual bottlenecks)
#
# If the leakage effect appears in both CNNs, it is not an artefact of one
# design. The two from-scratch transformers separate "transformers need data"
# from "this particular transformer was too large for this dataset", and the
# pretrained transformer completes the 2x2: with it, family varies against the
# CNNs while pretraining is held fixed. The pretrained models do NOT share a training
# regime: the CNNs unfreeze their last 20 weight-bearing layers, while the ViT
# backbone exposes far fewer leaves so its count differs. Measured per run.
ARCHS = {
    "ViT-30 (from scratch)": {"build": lambda: build_vit(30), "size": 30,
                              "epochs": 60, "batch": 32, "lr": 1e-3},
    "ViT-small (from scratch)": {"build": lambda: build_small_vit(72), "size": 72,
                                 "epochs": 60, "batch": 32, "lr": 1e-3},
    "ViT-B/16 (ImageNet pretrained)": {"build": lambda: build_pretrained_vit(224),
                                       "size": 224, "epochs": 30, "batch": 16,
                                       "lr": 1e-4, "optional": True},
    "EfficientNetB0 (pretrained)": {"build": lambda: build_effnet(224), "size": 224,
                                    "epochs": 30, "batch": 16, "lr": 1e-4},
    "ResNet50 (pretrained)": {"build": lambda: build_resnet(224), "size": 224,
                              "epochs": 30, "batch": 16, "lr": 1e-4},
}

In [ ]:
CACHE = {}
def images_at(size):
    if size not in CACHE:
        X = np.empty((len(df), size, size, 3), np.float32)
        for i, p in enumerate(tqdm(df["path"], desc=f"load {size}px", leave=False)):
            X[i] = np.asarray(Image.open(p).convert("RGB").resize((size, size),
                                                                  Image.BILINEAR),
                              np.float32)
        CACHE[size] = X
    return CACHE[size]

def evaluate(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0)
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1.mean()),
            "f1_benign": float(f1[0]), "f1_malignant": float(f1[1]),
            "f1_normal": float(f1[2])}

## 4. The factorial run

2 protocols x 5 architectures x 5 folds. Class weights are computed from each fold's
own training partition, never from the whole dataset, which would itself leak.

In [ ]:
rows = []
t_start = time.time()

skipped = {}
arch_params = {}      # so the write-up can state each arm's training regime
                      # from a measurement rather than from the source code
for arch_name, cfg in ARCHS.items():
    if cfg.get("optional"):
        # verify it can be built before spending time loading images for it
        try:
            keras.backend.clear_session()
            _probe = cfg["build"]()
            del _probe
            gc.collect()
            print(f"{arch_name}: pretrained weights available, including it")
        except Exception as e:
            skipped[arch_name] = f"{type(e).__name__}: {e}"
            print(f"{arch_name}: SKIPPED - {type(e).__name__}: {e}")
            print("  (reported as skipped; not silently replaced by another model)")
            continue
    X = images_at(cfg["size"])
    for proto_name, folds in PROTOCOLS.items():
        for k, (tr, te) in enumerate(folds):
            keras.backend.clear_session()
            tf.random.set_seed(SEED + k)

            counts = np.bincount(y_all[tr], minlength=3)
            cw = {i: float(len(tr) / (3 * c)) if c else 0.0
                  for i, c in enumerate(counts)}

            model = cfg["build"]()
            if arch_name not in arch_params:
                tp = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
                ap = sum(int(np.prod(w.shape)) for w in model.weights)
                arch_params[arch_name] = {"trainable": tp, "total": ap,
                                          "frac_trainable": round(tp / ap, 4)}
                print(f"  {arch_name}: {tp:,}/{ap:,} params trainable "
                      f"({100 * tp / ap:.1f}%)")
            model.compile(optimizer=keras.optimizers.Adam(cfg["lr"]),
                          loss="sparse_categorical_crossentropy",
                          metrics=["accuracy"])
            t0 = time.time()
            model.fit(X[tr], y_all[tr], epochs=cfg["epochs"], batch_size=cfg["batch"],
                      class_weight=cw, verbose=0)
            m = evaluate(y_all[te], model.predict(X[te], verbose=0).argmax(1))
            m.update({"architecture": arch_name, "protocol": proto_name, "fold": k,
                      "n_train": int(len(tr)), "n_test": int(len(te)),
                      "minutes": (time.time() - t0) / 60})
            rows.append(m)
            print(f"{arch_name[:22]:22s} | {proto_name:11s} | fold {k} | "
                  f"acc {m['accuracy']:.3f}  bal {m['balanced_accuracy']:.3f}  "
                  f"F1 {m['macro_f1']:.3f}  ({m['minutes']:.1f}m)")
            del model
            gc.collect()
    del CACHE[cfg["size"]]
    gc.collect()

cv = pd.DataFrame(rows)
cv.to_csv(f"{RESULTS_DIR}/cv_per_fold.csv", index=False)
RESULTS["skipped_architectures"] = skipped
RESULTS["architecture_params"] = arch_params
if skipped:
    print("\nSKIPPED:", skipped)
print(f"\ntotal {(time.time()-t_start)/60:.1f} min over {len(cv)} runs")
RESULTS["per_fold"] = cv.to_dict("records")
save_json()

## 5. Cross-validated performance

In [ ]:
summary = (cv.groupby(["architecture", "protocol"])
             .agg(acc_mean=("accuracy", "mean"), acc_std=("accuracy", "std"),
                  bal_mean=("balanced_accuracy", "mean"),
                  bal_std=("balanced_accuracy", "std"),
                  f1_mean=("macro_f1", "mean"), f1_std=("macro_f1", "std"))
             .round(4).reset_index())
print(summary.to_string(index=False))
summary.to_csv(f"{RESULTS_DIR}/cv_summary.csv", index=False)
RESULTS["summary"] = summary.to_dict("records")
save_json()

## 6. The leakage effect, per architecture

For each architecture, the difference between the two protocols' mean scores, with a
95\% confidence interval from Welch's t-test on the two sets of five folds. A positive
difference means the image-level protocol scored **higher**, which is the direction
leakage predicts.

In [ ]:
from itertools import combinations

def holm(pvals):
    # Holm-Bonferroni, NaN-safe, written out rather than imported. statsmodels
    # is not in this notebook's pip list, and an ImportError here would land
    # AFTER ~3 hours of training and destroy the entire analysis section. It is
    # nine lines; the dependency is not worth the failure mode. Matches
    # paper/recompute_stats.py exactly.
    p = np.asarray(pvals, dtype=float)
    out = np.full_like(p, np.nan)
    idx = np.where(~np.isnan(p))[0]
    order = idx[np.argsort(p[idx])]
    m, running = len(order), 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i])
        out[i] = min(1.0, running)
    return out

def perm_p(x, y):
    # Exact two-sided permutation test. With 5 vs 5 folds there are only
    # C(10,5) = 252 assignments, so the smallest attainable p is 2/252 = 0.0079.
    # Any Welch p far below that is an artefact of assuming normality on n=5,
    # which cannot be checked at n=5. Reporting this floor keeps the paper from
    # quoting precision the design cannot produce.
    pool, n, obs = np.concatenate([x, y]), len(x), abs(x.mean() - y.mean())
    idx = range(len(pool))
    diffs = [abs(pool[list(c)].mean() - pool[list(set(idx) - set(c))].mean())
             for c in combinations(idx, n)]
    return float((np.sum(np.array(diffs) >= obs - 1e-12)) / len(diffs))

effects = []
for arch in cv["architecture"].unique():
    a = cv[(cv.architecture == arch) & (cv.protocol == "image-level")]
    b = cv[(cv.architecture == arch) & (cv.protocol == "grouped")]
    for metric in ["accuracy", "balanced_accuracy", "macro_f1"]:
        x, y = a[metric].to_numpy(), b[metric].to_numpy()
        d = x.mean() - y.mean()
        t, p = stats.ttest_ind(x, y, equal_var=False)
        vx, vy = x.var(ddof=1) / len(x), y.var(ddof=1) / len(y)
        se = np.sqrt(vx + vy)
        dfree = (vx + vy)**2 / (vx**2/(len(x)-1) + vy**2/(len(y)-1)) if se else np.nan
        crit = stats.t.ppf(0.975, dfree) if se else np.nan

        # Nadeau-Bengio correction. Cross-validation folds are NOT independent
        # samples: any two training partitions share (k-2)/(k-1) of their data,
        # so var(ddof=1)/k understates the variance of the mean and an
        # uncorrected t-test is anticonservative. The standard remedy inflates
        # the variance by (1/k + n_test/n_train).
        # Use Welch-Satterthwaite df, NOT pooled 2k-2. The two arms' variances
        # differ by up to an order of magnitude (the grouped arm is far
        # noisier), so pooling understates the uncertainty exactly where it is
        # largest. This must match paper/recompute_stats.py: an earlier version
        # used pooled df here and Welch df there, and the same data then gave
        # 5/14 significant in one place and 3/14 in the other.
        rho = float(a["n_test"].mean() / a["n_train"].mean())
        infl = 1.0 + len(x) * rho
        se_nb = np.sqrt(infl * vx + infl * vy)
        t_nb = d / se_nb if se_nb else np.nan
        p_nb = float(2 * stats.t.sf(abs(t_nb), dfree)) if se_nb else np.nan
        crit_nb = stats.t.ppf(0.975, dfree) if se_nb else np.nan

        effects.append({"architecture": arch, "metric": metric,
                        "image_level": round(float(x.mean()), 4),
                        "grouped": round(float(y.mean()), 4),
                        "difference": round(float(d), 4),
                        "ci95_low": round(float(d - crit * se), 4),
                        "ci95_high": round(float(d + crit * se), 4),
                        "p_value": round(float(p), 5),
                        "p_permutation": round(perm_p(x, y), 5),
                        "ci95_nb_low": round(float(d - crit_nb * se_nb), 4),
                        "ci95_nb_high": round(float(d + crit_nb * se_nb), 4),
                        "p_nadeau_bengio": round(p_nb, 5)})

eff = pd.DataFrame(effects)

# Multiple comparisons: 5 architectures x 3 metrics = 15 tests, all reported.
# Holm over the corrected p-values, computed on the rows where a test was
# actually defined (a zero-variance arm yields no test, not a null result).
eff["p_nb_holm"] = np.round(holm(eff["p_nadeau_bengio"].to_numpy()), 5)
eff["significant_corrected"] = eff["p_nb_holm"] < 0.05

print(eff[["architecture", "metric", "difference", "p_value",
           "p_permutation", "p_nadeau_bengio", "p_nb_holm",
           "significant_corrected"]].to_string(index=False))
print()
print("p_value        : Welch, uncorrected. Reported for continuity only.")
print("p_permutation  : exact; FLOOR is 2/C(10,5) = 0.0079 by design.")
print("p_nadeau_bengio: corrects for the overlap between CV training sets.")
print("p_nb_holm      : the above, Holm-corrected across all 15 tests.")
print("The last column is the one the paper should quote.")
eff.to_csv(f"{RESULTS_DIR}/leakage_effect.csv", index=False)
with open(f"{RESULTS_DIR}/leakage_effect.md", "w") as f:
    f.write(eff.to_markdown(index=False))
RESULTS["leakage_effect"] = eff.to_dict("records")
save_json()

print()
print("Reading: a positive difference whose 95% CI excludes zero indicates the")
print("image-level protocol inflates the score. If the effect holds for ALL")
print("learning architectures, it is a property of the benchmark, not of one model.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, metric in zip(axes, ["accuracy", "balanced_accuracy", "macro_f1"]):
    labels, pos = [], 0
    for arch in cv["architecture"].unique():
        for proto, colour in [("image-level", "tab:red"), ("grouped", "tab:blue")]:
            v = cv[(cv.architecture == arch) & (cv.protocol == proto)][metric]
            ax.errorbar(pos, v.mean(), yerr=v.std(), fmt="o", capsize=4, color=colour)
            ax.scatter([pos] * len(v), v, alpha=.35, s=14, color=colour)
            labels.append(f"{arch.split()[0][:8]}\n{proto}"); pos += 1
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=7)
    ax.set_title(metric); ax.grid(alpha=.3, axis="y"); ax.set_ylim(0, 1)
plt.suptitle(f"{N_FOLDS}-fold cross-validation: image-level (red) vs grouped (blue)",
             fontsize=10)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/fig_protocol_effect.png", dpi=150)
plt.show()

In [ ]:
tbl = summary.copy()
tbl["accuracy"] = tbl.apply(lambda r: f"{r.acc_mean:.3f} +/- {r.acc_std:.3f}", axis=1)
tbl["balanced acc"] = tbl.apply(lambda r: f"{r.bal_mean:.3f} +/- {r.bal_std:.3f}", axis=1)
tbl["macro-F1"] = tbl.apply(lambda r: f"{r.f1_mean:.3f} +/- {r.f1_std:.3f}", axis=1)
final = tbl[["architecture", "protocol", "accuracy", "balanced acc", "macro-F1"]]
print(final.to_string(index=False))
with open(f"{RESULTS_DIR}/cv_table.md", "w") as f:
    f.write(final.to_markdown(index=False))
RESULTS["final_table"] = final.to_dict("records")
save_json()

print(sorted(os.listdir(RESULTS_DIR)))
deliver(f"{WORK}/fyp_phase2x_bundle", RESULTS_DIR)

## 7. Output and validity checks

`fyp_phase2x_bundle.zip` contains per-fold scores, the cross-validated summary, the
leakage effect with confidence intervals, and the comparison figure.

Conditions that invalidate the run:

- **The grouped protocol must show zero shared groups per fold**, asserted in section 2.
  If it does not, the "clean" arm is not clean and neither number means anything.
- **Class weights must come from each fold's own training partition.** Computing them
  over the whole dataset would leak test-set class distribution into training.
- **No early stopping and no model selection.** Every configuration is fixed in advance
  and every result is reported, so there is no selection effect to correct for.

Interpreting the outcome:

- **A positive difference for both architectures, with confidence intervals excluding
  zero**, supports the claim that image-level partitioning inflates results on this
  benchmark irrespective of model.
- **An effect for only one architecture** is a weaker and more interesting result, and
  is reported as such rather than generalised.
- **No effect** would contradict the Phase 0 estimate. In that case the Phase 0 number
  was measuring test-set difficulty rather than leakage, and this notebook's design is
  the trustworthy one. That outcome gets reported too.